In [ ]:

import os
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel

from datasets import load_dataset

import datasets, sys
print(f"versão dataset: {datasets.__version__}")

#### Função `average_pool`

<span style="font-size: 0.85em">

Reduz a saída token-a-token do modelo a um único vetor por sentença, ignorando os tokens de padding.

#### Parâmetros

**`last_hidden_states`** — tensor de saída da última camada do modelo, com shape `(batch, seq_len, hidden)`.

**`attention_mask`** — tensor que define quais tokens são reais (valor `1`) e quais devem ser ignorados (valor `0`). Shape `(batch, seq_len)`: uma linha por sequência do batch, uma posição por token.

#### Conceitos

**Tensor** — array multidimensional de números.

**Shape** — tupla com um valor por dimensão, indicando o tamanho de cada uma:

| Dimensões | Exemplo | Shape |
|---|---|---|
| 0 (escalar) | `3.14` | `()` |
| 1 (vetor) | `[1, 2, 3]` | `(3,)` |
| 2 (matriz) | `[[1, 2], [3, 4]]` | `(2, 2)` |
| 3 (cubo) | `[[[1,2],[3,4]], [[5,6],[7,8]]]` | `(2, 2, 2)` |
| N | sem nome próprio, mesma ideia | — |

#### Como funciona

Cada token vira um vetor de tamanho `hidden` (768, 1024 — depende do modelo), e a média é tirada **posição a posição**: a dimensão 0 do vetor final é a média das dimensões 0 de todos os tokens, e assim por diante.

O resultado tem exatamente o mesmo tamanho de um vetor de token individual, mas agora representa a sentença inteira.

</span>

In [ ]:
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

#### Carregamento do modelo

<span style="font-size: 0.85em">

```python
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')
```

**`AutoTokenizer`** — converte texto em IDs numéricos. Devolve `input_ids` (os tokens) e `attention_mask` (quais posições são reais).

**`AutoModel`** — a rede em si, sem cabeça de tarefa. Devolve `last_hidden_state` com shape `(batch, seq_len, 768)`.

**`Auto*`** — classes genéricas: leem a configuração do repositório e instanciam a arquitetura correta (aqui, XLM-RoBERTa) sem que você precise nomeá-la.

**`from_pretrained(...)`** — baixa os pesos do Hugging Face Hub na primeira execução e os guarda em cache local; nas seguintes, carrega do disco.

**`multilingual-e5-base`** — modelo de embeddings multilíngue (inclui português), `hidden = 768`. Usa mean pooling, daí a `average_pool`.
</span>

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')

#### Carregamento e tokenização dos documentos

<span style="font-size: 0.85em">

#### `load_dataset`

**`'unicamp-dl/mmarco'`** — versão multilíngue do MS MARCO, corpus padrão de recuperação de documentos.

**`'collection-portuguese'`** — a *config*: seleciona a coleção de passagens em português. É o acervo de textos, não muda entre treino e teste.

**`streaming=True`** — não baixa o dataset inteiro (são milhões de passagens); os exemplos chegam sob demanda. O retorno é um `IterableDataset`, que não aceita indexação por posição.

**`trust_remote_code=True`** — autoriza a execução do script `mmarco.py` do repositório. Exige `datasets < 4.0`; nas versões novas, loading scripts foram removidos.

#### Amostragem

**`.take(100)`** — pega os 100 primeiros exemplos do stream. O `list(...)` materializa o iterador em memória.

**`item['text']`** — extrai só o texto de cada registro, descartando o `id`.

#### `tokenizer`

**`max_length=512`** — limite de tokens por sequência.

**`truncation=True`** — corta o que passar desse limite.

**`padding=True`** — completa as sequências curtas até o comprimento da maior do batch, para formar um tensor retangular. É o que cria os paddings que a `average_pool` precisa ignorar.

**`return_tensors='pt'`** — devolve tensores PyTorch em vez de listas.

O resultado é um dicionário com `input_ids` e `attention_mask`, ambos com shape `(100, seq_len)`.

</span>

#### Carregamento e tokenização das consultas

<span style="font-size: 0.85em">

**`'queries-portuguese'`** — config com as consultas do MS MARCO, as queries são as perguntas para as "passage".

**`['train']`** — acessa os dados definidos para treino. Datasets costumam vir separados em train, validation e test para que treino e avaliação usem dados distintos.

**Prefixo `"query: "`** — obrigatório no E5, e distinto do `"passage: "` usado nos documentos. É assim que o modelo diferencia os dois papéis; sem isso o ranking degrada.

**`max_length=512`** — herdado do batch de documentos, mas consultas são curtas (poucas palavras). Como `padding=True` preenche até a maior sequência *do batch*, e não até 512, não há desperdício.

O resultado tem a mesma estrutura do `docs_batch`: `input_ids` e `attention_mask` com shape `(100, seq_len)`.

</span>

In [ ]:
dataset_docs = load_dataset('unicamp-dl/mmarco', 'collection-portuguese', streaming=True, trust_remote_code=True)
# Pega só os primeiros 100 exemplos
docs = list(dataset_docs['collection'].take(100))
# Separa o texto. O E5 exige o "passage: "
collection_textos = [f"passage: {item['text']}" for item in docs]

docs_batch = tokenizer(collection_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')

In [ ]:
dataset_queries = load_dataset('unicamp-dl/mmarco', 'queries-portuguese', streaming=True, trust_remote_code=True)
# Pega só os primeiros 100 exemplos
queries = list(dataset_queries['train'].take(100))
# Separa o texto. O E5 exige o "query: "
queries_textos = [f"query: {item['text']}" for item in queries]

queries_batch = tokenizer(queries_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')

In [ ]:
queries_outputs = model(**queries_batch)
queries_embeddings = average_pool(queries_outputs.last_hidden_state, queries_batch['attention_mask'])
# Normaliza os vetores (Normalização L2) para que todos tenham um "comprimento" igual a 1
queries_embeddings = F.normalize(queries_embeddings, p=2, dim=1)

print(queries_embeddings)

In [ ]:
docs_outputs = model(**docs_batch)
docs_embeddings = average_pool(docs_outputs.last_hidden_state, docs_batch['attention_mask'])

docs_embeddings = F.normalize(docs_embeddings, p=2, dim=1)

print(docs_embeddings)